In [ ]:
#default_exp en_task.en_task1

In [ ]:
#export
import os

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import transformers

import Hasoc.config as config
import Hasoc.utils.utils as utils
import Hasoc.utils.engine as engine
import Hasoc.model.model as model
import Hasoc.dataset.dataset as dataset

from functools import partial
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from transformers import AdamW, get_linear_schedule_with_warmup

In [ ]:
#export
SEED = 42
utils.seed_everything(SEED)

In [ ]:
#export
df = pd.read_csv(config.DATA_PATH/'fold_df.csv')

In [ ]:
#hide
df.head(2)

In [ ]:
#hide
df.shape

In [ ]:
#export
le = LabelEncoder()
le.fit_transform(df.task1)
le.classes_

In [ ]:
#export
df['task1_encoded'] = le.transform(df.task1.values)

In [ ]:
fold = 3
train_df = df.query(f'kfold_task1!={fold}').reset_index(drop=True)
valid_df = df.query(f'kfold_task1=={fold}').reset_index(drop=True)

In [ ]:
#export
train_dl = utils.create_loader(train_df.text.values, train_df.task1_encoded, bs=config.TRAIN_BATCH_SIZE)
valid_dl = utils.create_loader(valid_df.text.values, valid_df.task1_encoded, bs=config.VALID_BATCH_SIZE)

In [ ]:
train_dl.batch_size, valid_dl.batch_size

In [ ]:
#hide
next(iter(train_dl))

In [ ]:
#export
modeller = model.HasocModel(len(le.classes_))

In [ ]:
#export
model_params = list(modeller.named_parameters())

In [ ]:
#export
# we don't want weight decay for these
no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']

optimizer_params = [
    {'params': [p for n, p in model_params if n not in no_decay],
    'weight_decay':0.001},
    #  no weight decay should be applied
    {'params': [p for n, p in model_params if n in no_decay],
    'weight_decay':0.0}
]

In [ ]:
#export
lr = config.LR

In [ ]:
#export
optimizer = AdamW(optimizer_params, lr=lr)

In [ ]:
#export
num_train_steps = int(len(df) / config.TRAIN_BATCH_SIZE * config.NUM_EPOCHS)

In [ ]:
#export
scheduler = get_linear_schedule_with_warmup(optimizer=optimizer,
                                                num_warmup_steps=0,
                                                num_training_steps=num_train_steps)

In [ ]:
#export
fit = engine.BertFitter(modeller, (train_dl, valid_dl), optimizer, nn.CrossEntropyLoss(), partial(f1_score, average='macro'), config.DEVICE, scheduler=scheduler, log_file='en_task1_log.txt')
# fit = engine.BertFitter(modeller, (train_dl, valid_dl), optimizer, utils.LabelSmoothingCrossEntropy(), partial(f1_score, average='macro'), config.DEVICE, scheduler=scheduler, log_file='en_task1_log.txt')

In [ ]:
#export
NUM_EPOCHS = 6
fit.fit(NUM_EPOCHS, model_path=os.path.join(config.MODEL_PATH/'en_task1.pth'))

## inference

In [ ]:
#export
test_df = pd.read_csv(config.DATA_PATH/'en_task_a/english_test.csv')

In [ ]:
#hide
test_df.head()

In [ ]:
#export
test_dl = utils.create_loader(test_df.text.values, lbls=[None]*len(test_df.text.values), bs=config.VALID_BATCH_SIZE, is_test=True)

In [ ]:
#export
modeller = model.HasocModel(len(le.classes_))
modeller.load_state_dict(torch.load(config.MODEL_PATH/'en_task1.pth'))

In [ ]:
#export
preds = engine.get_preds(test_dl.dataset, test_dl, modeller, config.DEVICE)

In [ ]:
preds

In [ ]:
preds = le.inverse_transform(preds)

In [ ]:
preds

## Submission

In [ ]:
#export
sub = pd.read_csv(config.DATA_PATH/'en_task_a/english_test.csv')

In [ ]:
#export
submission_en_task1_df = test_df.drop(columns=['text', 'task1', 'task2']).copy()

In [ ]:
#export
submission_en_task1_df['task1'] = preds

In [ ]:
#export
submission_en_task1_df.to_csv(os.path.join('..', 'outputs', 'submission_EN_A.csv'), index=False)